# hadronic losses and gamma rays

In [1]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scipy.constants import k as k_B_SI
from matplotlib.colors import LogNorm

# Boltzmann constant in cgs: erg / K
k_B = k_B_SI * 1e7
k_B = 1.380649e-16  # erg K^-1

# seconds per Myr
Myr = 1e6 * 365.25 * 24 * 3600

from astropy import units as u, constants  as c

pc = c.pc.cgs.value
kB  = c.k_B.cgs.value
Msun = c.M_sun.cgs.value
G = c.G.cgs.value
Myr = u.Myr.to("s")
mp = c.m_p.cgs.value

# define CR energy fields
## first simple fields
constant CR energy and Gaussian vertical profile

In [2]:
EV_PER_CC_TO_ERG_PER_CM3 = 1.602176634e-12

def ecr_uniform(shape, ecr_eVcc=1.0):
    return np.full(shape, ecr_eVcc * EV_PER_CC_TO_ERG_PER_CM3)


#def ecr_stratified(z_pc, ecr0_eVcc=1.0, H_pc=1000.0):
#    return ecr0_eVcc * EV_PER_CC_TO_ERG_PER_CM3 * np.exp(-np.abs(z_pc) / H_pc)

def ecr_stratified(shape, z_pc, ecr0_eVcc=1.0, H_pc=1000.0):

    profile = (
        ecr0_eVcc
        * EV_PER_CC_TO_ERG_PER_CM3
        * np.exp(-np.abs(z_pc)/H_pc)
    )

    return np.broadcast_to(
        profile[None, None, :],
        shape
    )

def add_gaussian_blobs(ecr, x_pc, y_pc, z_pc, sn_list, amp_eVcc=1.0, sigma_pc=20.0):
    out = ecr.copy()
    amp = amp_eVcc * EV_PER_CC_TO_ERG_PER_CM3

    for x0, y0, z0 in sn_list:
        r2 = (x_pc-x0)**2 + (y_pc-y0)**2 + (z_pc-z0)**2
        out += amp * np.exp(-0.5 * r2 / sigma_pc**2)

    return out

## correlation with the density

In [3]:
def correlate_with_density(ecr_bg, nH, alpha=0.2, inverse=False):
    """
    alpha > 0 gives weak correlation strength.
    inverse=False: ecr increases with density
    inverse=True:  ecr decreases with density
    """
    n_norm = nH / np.nanmedian(nH)

    if inverse:
        factor = n_norm**(-alpha)
    else:
        factor = n_norm**alpha

    # preserve mean CR energy approximately
    factor /= np.nanmean(factor)

    return ecr_bg * factor

## streaming related model
in neutral gas, most of the waves are damped, CR stream freely, so the CR energy is mainly the background. In ionised gas, they couple and we implement a local Gaussian enhancement

In [4]:
def ecr_ionization_model(
    ecr_bg,
    x_pc, y_pc, z_pc,
    sn_list,
    f_ion,
    fion0=0.1,
    amp_eVcc=1.0,
    sigma_pc=30.0,
):
    """
    Neutral gas: fast transport -> mostly background.
    Ionized gas: stronger coupling -> local Gaussian CR enhancements.
    """
    coupling = f_ion / (f_ion + fion0)
    local = np.zeros_like(ecr_bg)

    amp = amp_eVcc * EV_PER_CC_TO_ERG_PER_CM3

    for x0, y0, z0 in sn_list:
        r2 = (x_pc-x0)**2 + (y_pc-y0)**2 + (z_pc-z0)**2
        local += amp * np.exp(-0.5 * r2 / sigma_pc**2)

    return ecr_bg + coupling * local

## gamma emissivity
This function scales with the gas number density and the CR energy density. Prefactors need to be added, see Isabelle's slides..

In [5]:
def gamma_emissivity_proxy(nH, ecr):
    """
    Proxy emissivity, proportional to nH * ecr.
    Units are arbitrary unless calibrated.
    """
    return nH * ecr


def gamma_luminosity_proxy(nH, ecr, cell_volume_cm3):
    q = gamma_emissivity_proxy(nH, ecr)
    return np.sum(q * cell_volume_cm3)

In [6]:
def hadronic_cooling_time(nH, sigma_pp=3e-26, K=0.5):
    c = 2.99792458e10
    t_sec = 1.0 / (nH * sigma_pp * c * K)
    return t_sec / (3600 * 24 * 365.25 * 1e6)  # Myr

## clumping factor

In [7]:
def gamma_clumping_factor(nH, ecr):
    return np.nanmean(nH * ecr) / (np.nanmean(nH) * np.nanmean(ecr))

In [8]:
def block_reduce_mean(arr, block):
    nx, ny, nz = arr.shape
    nx2, ny2, nz2 = nx//block, ny//block, nz//block

    arr_cut = arr[:nx2*block, :ny2*block, :nz2*block]

    return arr_cut.reshape(
        nx2, block,
        ny2, block,
        nz2, block
    ).mean(axis=(1,3,5))


def block_clumping_analysis(nH, ecr, block_sizes):
    results = {}

    for b in block_sizes:
        n_b = block_reduce_mean(nH, b)
        e_b = block_reduce_mean(ecr, b)
        q_b = block_reduce_mean(nH * ecr, b)

        C_b = q_b / (n_b * e_b)

        results[b] = {
            "C_mean": np.nanmean(C_b),
            "C_median": np.nanmedian(C_b),
            "C_16": np.nanpercentile(C_b, 16),
            "C_84": np.nanpercentile(C_b, 84),
            "C_field": C_b,
        }

    return results

In [9]:
# load SILCC data cube

with open('../sim-data/SILCC_hdf5_plt_cnt_3000-uniform-cube-N256-full.pkl', 'rb') as handle:
    data = pickle.load(handle)

#with open('../tmp-data/scaled-density-Teq-cube.pkl', 'rb') as handle:
#    data = pickle.load(handle)
recent_SNe = np.loadtxt("../sim-data/recent_SNe.txt")
#print(data["x_pc"])

In [10]:
models = {}

models["uniform_0p1"] = ecr_uniform(data["n"].shape, 0.1)
models["uniform_1"]   = ecr_uniform(data["n"].shape, 1.0)
models["uniform_10"]  = ecr_uniform(data["n"].shape, 10.0)

models["strat_H100"] = ecr_stratified(
    data["n"].shape,
    data["z_pc"],
    ecr0_eVcc=1.0,
    H_pc=100.0,
)

models["strat_H1000"] = ecr_stratified(
    data["n"].shape,
    data["z_pc"],
    ecr0_eVcc=1.0,
    H_pc=1000.0,
)

In [11]:
#models["strat_H100"] = ecr_stratified(data["z_pc"], 1.0, H_pc=100.0)
#models["strat_H1000"] = ecr_stratified(data["z_pc"], 1.0, H_pc=1000.0)
if True:
    bg = models["strat_H1000"]
    
    models["SN_blobs"] = add_gaussian_blobs(
        bg, data["x_pc"], data["y_pc"], data["z_pc"], recent_SNe[:,1:4],
        amp_eVcc=3.0,
        sigma_pc=30.0,
    )
    
    models["density_corr"]     = correlate_with_density(bg, data["n"], alpha=0.2)
    models["density_anticorr"] = correlate_with_density(bg, data["n"], alpha=0.2, inverse=True)
    
    models["ionization_streaming"] = ecr_ionization_model(
        bg, data["x_pc"], data["y_pc"], data["z_pc"],
        recent_SNe[:,1:4],
        data["f_ion"],
        fion0=0.1,
        amp_eVcc=3.0,
        sigma_pc=30.0,
    )

In [12]:
cell_volume_pc3 = (data["x_pc"][1]-data["x_pc"][0])**3
cell_volume_cm3 = cell_volume_pc3 * pc**3

model_results = {}

emap=""
cmap="plasma"

vmin=1e-10
vmax=1e-8

extent = [data["x_pc_bnds"][0], data["x_pc_bnds"][-1], data["x_pc_bnds"][0], data["x_pc_bnds"][-1]]

for name, ecr in models.items():
    model_results[name] = {}
    
    qgamma = gamma_emissivity_proxy(data["n"], ecr)
    Lgamma = gamma_luminosity_proxy(data["n"], ecr, cell_volume_cm3)
    Cgamma = gamma_clumping_factor(data["n"], ecr)
    thad = hadronic_cooling_time(data["n"])

    model_results[name]["qgamma"] = qgamma
    model_results[name]["Lgamma"] = Lgamma
    model_results[name]["thad"]   = thad
    model_results[name]["Cgamma"] = Cgamma
    
    print(name)
    print("  Lgamma proxy =", Lgamma)
    print("  Cgamma       =", Cgamma)
    print("  t_had median =", np.nanmedian(thad), "Myr")
    print()

    fig, ax = plt.subplots(figsize=(10,4), ncols=3, sharey=True)
    im = ax[0].imshow(np.sum(ecr, axis=0).T, norm=LogNorm(vmin=vmin, vmax=vmax), extent=extent, origin="lower")
    im = ax[1].imshow(np.sum(ecr, axis=1).T, norm=LogNorm(vmin=vmin, vmax=vmax), extent=extent, origin="lower")
    im = ax[2].imshow(np.sum(ecr, axis=2),   norm=LogNorm(vmin=vmin, vmax=vmax), extent=extent, origin="lower")

    fig.subplots_adjust(right=0.8, wspace=0.05)
    cbar_ax = fig.add_axes([0.82, 0.25, 0.025, 0.5])
    fig.colorbar(im, cax=cbar_ax)


NameError: name 'thad' is not defined